# **Retail Sales ETL Pipeline**

This notebook demonstrates a simple **Extract, Transform, Load (ETL)** workflow using the Superstore retail sales dataset.

The goal of this project is to:
- load the raw sales dataset,
- clean and standardize the data,
- create smaller dimension tables,
- build a central fact table,
- validate that the transformed tables still match the original cleaned dataset,
- export the final tables as CSV files for future analysis or reporting.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **1. Data Extraction**

The first stage of the ETL process is extracting the raw dataset from the storage location.


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/Colab Notebooks/superstore_dataset.csv"
df = pd.read_csv(file_path)
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## **2. Initial Data Formatting**

The `Sales` column is rounded to two decimal places to make the values easier to read and more consistent with normal currency formatting.


In [3]:
df['Sales'] = df['Sales'].round(2)
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37


## **3. Initial Data Inspection**

Before transforming the dataset, it is important to understand the size and structure of the data.

The next few cells check:
- the number of rows and columns,
- the column data types,
- whether any columns need formatting or cleaning.


In [4]:
df.shape

(9800, 18)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

## **4. Column Name Standardization**

The original dataset contains column names with uppercase letters, spaces, and special characters. These are standardized into a cleaner format using lowercase text and underscores.

For example, `Order Date` becomes `order_date`.

Standardized column names make the notebook easier to read and reduce the chance of errors when selecting columns later.


In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales'],
      dtype='object')

## **5. Date Conversion**

The `order_date` and `ship_date` columns are converted into datetime format.

This is important because date columns need to be in datetime format before they can be used for time-based analysis, such as extracting year, month, or calculating shipping duration.


In [7]:
# Convert order_date and ship_date columns into datetime format
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

## **6. Missing Value Check**

This step checks for missing values in every column.

Missing values should be reviewed before creating dimension and fact tables because incomplete records may affect data quality and downstream analysis.


In [8]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

,0
postal_code,11


### Handling Missing Postal Codes

Rows with missing postal codes are removed because they would create incomplete location records.

This keeps the location table cleaner and avoids missing keys when building the final fact table.


In [9]:
df = df.dropna(subset=['postal_code'])

## **7. Duplicate Row Check**

Duplicate rows can cause incorrect totals, especially for measures such as sales.

This step checks whether the dataset contains fully duplicated rows before continuing with the ETL process.


In [10]:
# Check total number of duplicate rows
df.duplicated().sum()

np.int64(0)

## **8. Creating Date Features**

Additional date fields are created from `order_date`.

The new columns are:
- `year`
- `month`
- `year_month`

These fields make it easier to group sales by time period and are later used to build the date dimension table.


In [11]:
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)

## **9. Postal Code Formatting**

The postal code column is converted from float format into string format.

Postal codes are identifiers, not values used for mathematical calculations. Storing them as strings helps keep it as categorical location data.


In [12]:
# Convert Postal Code from float to integer, then to string
df['postal_code'] = df['postal_code'].astype(int).astype(str)

# Check result
df['postal_code'].head()

,postal_code
0,42420
1,42420
2,90036
3,33311
4,33311


## **10. Shipping Duration Analysis**

The `shipping_days` column calculates the number of days between the order date and ship date.

This creates a useful metric for checking delivery speed by shipping mode. The grouped summary helps confirm whether the shipping modes behave as expected.


In [13]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

df.groupby('ship_mode')['shipping_days'].describe()

,count,mean,std,min,25%,50%,75%,max
ship_mode,,,,,,,,
First Class,1501.0,2.179214,0.774076,1.0,2.0,2.0,3.0,4.0
Same Day,538.0,0.044610,0.206637,0.0,0.0,0.0,0.0,1.0
Second Class,1901.0,3.249868,1.188168,1.0,2.0,3.0,4.0,5.0
Standard Class,5849.0,5.009916,1.010667,3.0,4.0,5.0,6.0,7.0


## **11. Final Duplicate Check After Cleaning**

After the main cleaning steps, the duplicate check is repeated to confirm that the cleaned dataset is still suitable for transformation.

This is a simple data quality check before creating the dimension and fact tables.


In [14]:
df.duplicated().sum()

np.int64(0)

## **12. Data Profiling**

This loop prints the value counts for each column.

Data profiling helps identify:
- unusual values,
- inconsistent categories,
- duplicated identifiers,
- columns that may be useful for dimension tables.

This output can be long, but it is useful during the exploration stage of an ETL project.


In [15]:
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 50)


Column: row_id
row_id
9800    1
1       1
2       1
3       1
4       1
       ..
18      1
17      1
16      1
15      1
14      1
Name: count, Length: 9789, dtype: int64
--------------------------------------------------

Column: order_id
order_id
CA-2018-100111    14
CA-2018-157987    12
CA-2017-165330    11
US-2017-108504    11
CA-2017-105732    10
                  ..
CA-2018-131303     1
US-2016-100069     1
CA-2015-128062     1
CA-2018-124261     1
CA-2018-114440     1
Name: count, Length: 4916, dtype: int64
--------------------------------------------------

Column: order_date
order_date
2017-09-05    38
2017-11-10    35
2018-12-02    34
2018-12-01    34
2018-09-02    33
              ..
2016-04-14     1
2015-06-10     1
2016-01-28     1
2015-06-18     1
2016-05-09     1
Name: count, Length: 1229, dtype: int64
--------------------------------------------------

Column: ship_date
ship_date
2018-09-26    34
2018-12-06    32
2016-12-16    31
2018-09-15    30
2018-12-12    30
    

## **13. Column Review**

This step displays the final list of column names after cleaning and feature engineering.

Reviewing the available columns makes it easier to decide which columns should be placed into dimension tables and which columns should remain in the fact table.


In [16]:
list(df.columns)

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'year',
 'month',
 'year_month',
 'shipping_days']

# **14. Building Dimension Tables**

Dimension tables store descriptive information about business entities. They help reduce repeated values in the main transaction table and make the final data model easier to query.

In this project, the dimension tables are:
- `dim_customer`
- `dim_product`
- `dim_location`
- `dim_date`
- `dim_shipping`

Each dimension table is created by selecting relevant columns and removing duplicate combinations.

## 14.1 Customer Dimension

The customer dimension stores unique customer details such as customer ID, customer name, and segment.

Removing duplicates ensures that each customer appears only once in this dimension table.


In [17]:
dim_customer = df[[
    'customer_id',
    'customer_name',
    'segment'
]].drop_duplicates().reset_index(drop=True)

dim_customer.head()

,customer_id,customer_name,segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


## 14.2 Product Dimension

The product dimension stores product-related information such as product ID, product name, category, and sub-category.

At this stage, the table is created using the original `product_id`. Validation step checks later whether this ID is unique enough to be used as a product key.


In [18]:
dim_product = df[[
    'product_id',
    'product_name',
    'category',
    'sub_category'
]].drop_duplicates().reset_index(drop=True)

dim_product.head()

,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


## 14.3 Location Dimension

The location dimension stores geographical information such as postal code, country, city, state, and region.

Creating a separate location table avoids repeating the same location details in every sales transaction row.


In [19]:
dim_location = df[[
    'postal_code',
    'country',
    'city',
    'state',
    'region'
]].drop_duplicates().reset_index(drop=True)

dim_location.head()

,postal_code,country,city,state,region
0,42420,United States,Henderson,Kentucky,South
1,90036,United States,Los Angeles,California,West
2,33311,United States,Fort Lauderdale,Florida,South
3,90032,United States,Los Angeles,California,West
4,28027,United States,Concord,North Carolina,South


## 14.4 Date Dimension

The date dimension stores order date information and related date fields such as year, month, and year-month.

This table supports time-based analysis, such as monthly sales trends, yearly comparisons, and reporting by period.


In [20]:
dim_date = df[[
    'order_date',
    'year',
    'month',
    'year_month'
]].drop_duplicates().reset_index(drop=True)

dim_date.head()

,order_date,year,month,year_month
0,2017-11-08,2017,11,2017-11
1,2017-06-12,2017,6,2017-06
2,2016-10-11,2016,10,2016-10
3,2015-06-09,2015,6,2015-06
4,2018-04-15,2018,4,2018-04


## 14.5 Shipping Dimension

The shipping dimension stores the available shipping modes.

A new `ship_mode_id` is created as a simple surrogate key. This key can be used in the fact table instead of repeating the full shipping mode text in every row.


In [21]:
dim_shipping = df[[
    'ship_mode'
]].drop_duplicates().reset_index(drop=True)

dim_shipping['ship_mode_id'] = dim_shipping.index + 1

dim_shipping = dim_shipping[[
    'ship_mode_id',
    'ship_mode'
]]

dim_shipping.head()

,ship_mode_id,ship_mode
0,1,Second Class
1,2,Standard Class
2,3,First Class
3,4,Same Day


## 14.6 Adding a Location Key

A new `location_id` is created for the location dimension.

This surrogate key makes the location table easier to connect to the fact table and avoids using multiple location columns as a join key in future analysis.


In [22]:
dim_location['location_id'] = dim_location.index + 1

dim_location = dim_location[[
    'location_id',
    'postal_code',
    'country',
    'city',
    'state',
    'region'
]]

dim_location.head()

,location_id,postal_code,country,city,state,region
0,1,42420,United States,Henderson,Kentucky,South
1,2,90036,United States,Los Angeles,California,West
2,3,33311,United States,Fort Lauderdale,Florida,South
3,4,90032,United States,Los Angeles,California,West
4,5,28027,United States,Concord,North Carolina,South


# **15. Preparing the Data Model**

The cleaned main DataFrame is merged with the dimension tables so that the correct surrogate keys can be attached to each transaction row.

At this stage, `location_id` and `ship_mode_id` are added back into the transaction-level dataset.


In [23]:
df_model = df.merge(
    dim_location,
    on=['postal_code', 'country', 'city', 'state', 'region'],
    how='left'
)

df_model = df_model.merge(
    dim_shipping,
    on='ship_mode',
    how='left'
)

df_model.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,category,sub_category,product_name,sales,year,month,year_month,shipping_days,location_id,ship_mode_id
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2017,11,2017-11,3,1,1
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,2017,11,2017-11,3,1,1
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2017,6,2017-06,4,2,1
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,2016,10,2016-10,7,3,2
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2016,10,2016-10,7,3,2


## **16. Creating the Initial Fact Table**

The fact table contains the transaction-level sales records.

It keeps the measurable business values, such as `sales` and `shipping_days`, together with keys that connect each transaction to the dimension tables.

This first version still uses the original `product_id`.


In [24]:
fact_sales = df_model[[
    'row_id',
    'order_id',
    'order_date',
    'ship_date',
    'customer_id',
    'product_id',
    'location_id',
    'ship_mode_id',
    'sales',
    'shipping_days'
]].copy()

fact_sales.head()

,row_id,order_id,order_date,ship_date,customer_id,product_id,location_id,ship_mode_id,sales,shipping_days
0,1,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,FUR-BO-10001798,1,1,261.96,3
1,2,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,FUR-CH-10000454,1,1,731.94,3
2,3,CA-2017-138688,2017-06-12,2017-06-16,DV-13045,OFF-LA-10000240,2,1,14.62,4
3,4,US-2016-108966,2016-10-11,2016-10-18,SO-20335,FUR-TA-10000577,3,2,957.58,7
4,5,US-2016-108966,2016-10-11,2016-10-18,SO-20335,OFF-ST-10000760,3,2,22.37,7


## **17. Row Count Validation**

The number of rows in the fact table should match the number of rows in the cleaned original dataset.

If the row counts do not match, it may indicate that the merge process created duplicate rows or removed records.


In [25]:
print("Original dataframe rows:", df.shape[0])
print("Fact sales rows:", fact_sales.shape[0])

Original dataframe rows: 9789
Fact sales rows: 9789


## **18. Primary Key Validation for Dimension Tables**

This step checks whether the key columns in each dimension table contain duplicates.

A dimension key should normally be unique. Duplicate keys can create incorrect joins and duplicated records in the fact table.


In [26]:
print("Duplicate customer_id:", dim_customer['customer_id'].duplicated().sum())
print("Duplicate product_id:", dim_product['product_id'].duplicated().sum())
print("Duplicate location_id:", dim_location['location_id'].duplicated().sum())
print("Duplicate ship_mode_id:", dim_shipping['ship_mode_id'].duplicated().sum())

Duplicate customer_id: 0
Duplicate product_id: 32
Duplicate location_id: 0
Duplicate ship_mode_id: 0


## **19. Investigating Duplicate Product IDs**

The product dimension check shows whether the same `product_id` appears more than once.

This can happen when one product ID is linked to multiple product names, categories, or sub-categories. If this happens, the original `product_id` is not reliable as the unique key for the product dimension.


In [27]:
duplicated_product_ids = dim_product[
    dim_product['product_id'].duplicated(keep=False)
].sort_values('product_id')

duplicated_product_ids

,product_id,product_name,category,sub_category
1364,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture,Bookcases
1259,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture,Bookcases
64,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture,Chairs
124,FUR-CH-10001146,"Global Task Chair, Black",Furniture,Chairs
1010,FUR-FU-10001473,DAX Wood Document Frame,Furniture,Furnishings
...,...,...,...,...
890,TEC-PH-10002200,Samsung Galaxy Note 2,Technology,Phones
1399,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology,Phones
974,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology,Phones
727,TEC-PH-10004531,OtterBox Commuter Series Case - iPhone 5 & 5s,Technology,Phones


## **20. Creating a Product Surrogate Key**

To solve the duplicate `product_id` issue, a new `product_key` is created.

This `product_key` acts as a unique surrogate key for each unique product record. The original `product_id` is still kept as a business identifier, but the fact table will use `product_key` to avoid duplicate key problems.


In [28]:
dim_product = df[[
    'product_id',
    'product_name',
    'category',
    'sub_category'
]].drop_duplicates().reset_index(drop=True)

dim_product.insert(0, 'product_key', range(1, len(dim_product) + 1))

dim_product.head()

,product_key,product_id,product_name,category,sub_category
0,1,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,2,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,3,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,4,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,5,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


## **21. Validating the New Product Key**

This step confirms that the new `product_key` is unique.

It also checks the original `product_id` again to show the difference between a clean surrogate key and a business identifier that may not be fully unique.


In [29]:
print("Duplicate product_key:", dim_product['product_key'].duplicated().sum())
print("Duplicate product_id:", dim_product['product_id'].duplicated().sum())

Duplicate product_key: 0
Duplicate product_id: 32


## **22. Reconnecting Product Keys to the Main Dataset**

The updated product dimension is merged back into the main modelling DataFrame.

This attaches the correct `product_key` to each transaction row based on the full product details.


In [30]:
df_model = df_model.merge(
    dim_product,
    on=['product_id', 'product_name', 'category', 'sub_category'],
    how='left'
)

## **23. Final Fact Table**

The final fact table is rebuilt using `product_key` instead of the original `product_id`.

This produces a cleaner star schema because the fact table now connects to the product dimension through a unique product key.


In [31]:
fact_sales = df_model[[
    'row_id',
    'order_id',
    'order_date',
    'ship_date',
    'customer_id',
    'product_key',
    'location_id',
    'ship_mode_id',
    'sales',
    'shipping_days'
]].copy()

fact_sales.head()

,row_id,order_id,order_date,ship_date,customer_id,product_key,location_id,ship_mode_id,sales,shipping_days
0,1,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,1,1,1,261.96,3
1,2,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,2,1,1,731.94,3
2,3,CA-2017-138688,2017-06-12,2017-06-16,DV-13045,3,2,1,14.62,4
3,4,US-2016-108966,2016-10-11,2016-10-18,SO-20335,4,3,2,957.58,7
4,5,US-2016-108966,2016-10-11,2016-10-18,SO-20335,5,3,2,22.37,7


# **24. Data Validation**

Data validation is used to confirm that the ETL process produced reliable output tables.


The validation checks below confirm that:
- the fact table has the same number of rows as the cleaned dataset,
- there are no missing foreign keys in important columns,
- the transformation process did not accidentally remove or duplicate records.


In [32]:
print("Original cleaned dataframe rows:", df.shape[0])
print("Fact sales rows:", fact_sales.shape[0])

print("Missing customer_id in fact:", fact_sales['customer_id'].isnull().sum())
print("Missing product_key in fact:", fact_sales['product_key'].isnull().sum())
print("Missing location_id in fact:", fact_sales['location_id'].isnull().sum())
print("Missing ship_mode_id in fact:", fact_sales['ship_mode_id'].isnull().sum())

Original cleaned dataframe rows: 9789
Fact sales rows: 9789
Missing customer_id in fact: 0
Missing product_key in fact: 0
Missing location_id in fact: 0
Missing ship_mode_id in fact: 0


## **25. Sales Reconciliation**

Sales reconciliation compares the total sales value in the cleaned original dataset against the total sales value in the final fact table.

The difference should be zero. This confirms that the ETL transformation preserved the main sales measure correctly.


In [33]:
original_sales_total = df['sales'].sum()
fact_sales_total = fact_sales['sales'].sum()

print("Original sales total:", round(original_sales_total, 2))
print("Fact table sales total:", round(fact_sales_total, 2))
print("Difference:", round(original_sales_total - fact_sales_total, 2))

Original sales total: 2252607.18
Fact table sales total: 2252607.18
Difference: 0.0


# **26. Loading / Exporting the Final Tables**

The final stage of the ETL process is **loading** the transformed tables into an output location.

In this notebook, the dimension tables and fact table are exported as CSV files. These files can later be imported into SQL, Power BI, Tableau, or another analytics tool.


In [34]:
output_path = "/content/drive/MyDrive/Colab Notebooks/superstore_etl_output/"

dim_customer.to_csv(output_path + "dim_customer.csv", index=False)
dim_product.to_csv(output_path + "dim_product.csv", index=False)
dim_location.to_csv(output_path + "dim_location.csv", index=False)
dim_date.to_csv(output_path + "dim_date.csv", index=False)
dim_shipping.to_csv(output_path + "dim_shipping.csv", index=False)
fact_sales.to_csv(output_path + "fact_sales.csv", index=False)

print("All tables exported successfully.")

All tables exported successfully.


# **27. Example Analysis Using the Star Schema**

After the ETL process is completed, the final tables can be used for analysis.

The example below joins the fact table with the date dimension and calculates monthly sales. This demonstrates how analysts can use the transformed tables instead of relying only on the original raw dataset.


In [35]:
sales_by_month = fact_sales.merge(
    dim_date,
    left_on='order_date',
    right_on='order_date',
    how='left'
).groupby('year_month')['sales'].sum().reset_index()

sales_by_month.sort_values('year_month')

,year_month,sales
0,2015-01,14205.71
1,2015-02,4519.92
2,2015-03,55205.81
3,2015-04,27906.86
4,2015-05,23644.29
5,2015-06,34322.92
6,2015-07,33781.52
7,2015-08,27117.53
8,2015-09,81623.50
9,2015-10,31453.37
